# SPA-TTR examples

This notebook begins with the basic TTR -> SPA definitions discussed in [*TTR at the SPA*](https://aclanthology.org/2023.naloma-1.5/), then implements the question-answering example from [*Finding Answers to Questions: Bridging between Type-based and Computational Neuroscience Approaches*](https://aclanthology.org/2025.iwcs-main.11/).

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd()
if not (project_root / "src").is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from pyttr.categorical import BType, Fun, PType, Pred, RecType
from pyttr.spa import (
    AssociativeMemory, Codebook, Encoder, QuestionEncoder, SPAModel, WitnessCache,
)

## Mapping TTR to SPA

Following *TTR at the SPA*, let $\sigma$ map a TTR type or other structured TTR object to its corresponding SPA pattern of neural activity, represented as a hypervector. We will also write $T \sim \mathbf{T}$, for $\sigma(T) = \mathbf{T}$. 

For a basic type, $\sigma$ is supplied by a codebook of distinct semantic pointers:

$$\sigma(T) = \beta(T) \qquad \text{if } T \text{ is a basic type.}$$

For a labelled TTR structure, encoding is compositional. Each label $\ell_i$ receives the stable pointer $\beta(\ell_i)$ from the same codebook as basic types. The encoded structure binds each label pointer to its encoded value, then superposes the bindings:

$$\sigma(\{\langle \ell_1, x_1 \rangle, \ldots, \langle \ell_n, x_n \rangle\}) = \beta(\ell_1) \mathbin{\circledast} \sigma(x_1) + \ldots + \beta(\ell_n) \mathbin{\circledast} \sigma(x_n).$$

Thus, a record type is encoded as:

$$\sigma(\left[\begin{array}{lcl}\ell_1 & : & T_1\\ \vdots & \vdots & \vdots\\ \ell_n & : & T_n\end{array}\right]) = \beta(\ell_1) \mathbin{\circledast} \sigma(T_1) + \ldots + \beta(\ell_n) \mathbin{\circledast} \sigma(T_n).$$

A predicate type (PType) llis a labelled structure whose predicate and arguments have conventional roles:

$$\sigma(P(a_1, \ldots, a_n)) = pred \mathbin{\circledast} P + \sum_{i=1}^{n} arg_i \mathbin{\circledast} \sigma(a_i).$$

Where $pred$ and $arg_i, ... arg_N$ are arbitrary symbolic pointer vectors.

## Basic type judgements in an SPA model

In normal TTR, a judgement such as `alice : Ind` can be recorded in a type's witness cache. *TTR at the SPA* proposes an analogous vector structure: bind each basic type pointer to a superposition of its witness pointers, then bundle those bindings into a cache `F`. To test `alice : Ind`, unbind the `Ind` pointer from `F` and compare the noisy recovered bundle with the pointer for `alice`.

`SPAModel` combines the structural encoder $\sigma$, a `WitnessCache`, and a default decision threshold. `judge(a, T)` records the modeled fact $a : T$ in the witness memory. `query(a, T)` returns its continuous similarity evidence, while `judges(a, T)` compares that score with the model threshold and returns a Boolean decision. The threshold is a modeling decision: it depends on dimensionality, cache size, the VSA model, and the required tolerance for false positives. This example uses HRR to align with the paper, but the cache is also tested with TorchHD's `BSC`, `MAP`, and `FHRR` models.

In [2]:
Ind = BType("Ind")
Color = BType("Color")
spa_model = SPAModel(
    Codebook(dimensions=10_000, vsa="HRR", seed=7), threshold=0.2
)

# judge() registers atomic TTR facts in the model's SPA witness memory.
spa_model.judge("alice", Ind)
spa_model.judge("bob", Ind)
spa_model.judge("red", Color)

# query() returns a similarity score rather than a symbolic truth value.
evidence = {
    "alice : Ind": spa_model.query("alice", Ind),
    "carol : Ind": spa_model.query("carol", Ind),
    "red : Ind": spa_model.query("red", Ind),
    "red : Color": spa_model.query("red", Color),
}
# judges() turns evidence into a decision using the model's threshold (0.2).
assert spa_model.judges("alice", Ind)
assert not spa_model.judges("carol", Ind)
{name: round(score, 3) for name, score in evidence.items()}

{'alice : Ind': 0.437,
 'carol : Ind': -0.012,
 'red : Ind': 0.016,
 'red : Color': 0.486}

In [3]:
spa_model.members(Ind), spa_model.members(Color)

((('bob', 0.4558946490287781), ('alice', 0.4368399381637573)),
 (('red', 0.48571377992630005),))

## Record-aware SPA judgements

`SPAModel.query()` extends witness-cache judgment to a concrete record and a TTR record type. It resolves a dependent field against the values named in its dependency list, then checks the resulting type through the cache. Thus, for $[x : Ind, c_{man} : man(x)]$, a record with `x = alice` and `c_man = e_alice_man` requires evidence for both `alice : Ind` and `e_alice_man : man(alice)`. The overall score is the weakest required field score.

The record remains explicit while its dependent fields are resolved; the atomic judgments are the similarity-based SPA operations. This avoids treating an uncleaned, superposed record pointer as an exact symbolic object.

In [4]:
man = Pred("man", [Ind])
man_record_type = RecType({
    "x": Ind,
    "c_man": (Fun("v", Ind, PType(man, ["v"])), ["x"]),
})

# This registers the predicate witness needed after the dependent field resolves.
spa_model.judge("e_alice_man", PType(man, ["alice"]))
alice_record = {"x": "alice", "c_man": "e_alice_man"}
wrong_record = {"x": "bob", "c_man": "e_alice_man"}

# For a record type, judges() resolves dependencies before thresholding each field.
assert spa_model.judges(alice_record, man_record_type)
assert not spa_model.judges(wrong_record, man_record_type)
spa_model.explain(alice_record, man_record_type)

{'score': 0.3606971800327301,
 'fields': ({'label': 'x',
   'score': 0.3606971800327301,
   'expected': <pyttr.categorical.type_core.BTypeClass at 0x771a2cb0eba0>},
  {'label': 'c_man',
   'score': 0.3606971800327301,
   'expected': <pyttr.categorical.compound_types.PTypeClass at 0x771bb85c8cd0>})}

## Case study: Record Type Merge

The notation `c_man : man(x)` is a dependent field. In `pyttr`, it is a function over `x` together with the dependency label `x`.

In [15]:
Ind = BType("Ind")
Ind.judge("a")
Ind.judge("b")
man = Pred("man", [Ind])
run = Pred("run", [Ind])

man_at_x = (Fun("v", Ind, PType(man, ["v"])), ["x"])
man_at_y = (Fun("v", Ind, PType(man, ["v"])), ["y"])
run_at_x = (Fun("v", Ind, PType(run, ["v"])), ["x"])

man_type = RecType({"x": Ind, "c_man": man_at_x})
run_type = RecType({"x": Ind, "c_run": run_at_x})
man_type_2 = RecType({"y": Ind, "c_man": man_at_y})
man_rec_type = RecType({"c_man": Pred("man", ['a'])})
run_rec_type = RecType({"c_man": Pred("run", ['b'])})
merged_type = man_type.merge(run_type)

assert set(merged_type.comps.labels()) == {"x", "c_man", "c_run"}
merged_type.show()

'{x : Ind, c_man : (lambda v:Ind . man(v), [x]), c_run : (lambda v:Ind . run(v), [x])}'

In [16]:
from pyttr.utils import show_latex

display(show_latex(man_type))
display(show_latex(run_rec_type))
display(show_latex(man_rec_type))


display(show_latex(run_type))
display(show_latex(merged_type))

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

### SPA encoding

`Codebook` assigns stable random hypervectors to labels, type names, predicates, and variables. `Encoder` binds each record label to its field encoding and superposes the resulting bindings. The result is approximate: we compare cosine similarities rather than vector equality.

In [20]:
encoder = Encoder(Codebook(dimensions=10_000, seed=7))
merged_pointer = encoder.encode(merged_type)

scores = {
    "man component": encoder.similarity(merged_pointer, encoder.encode(man_type)),
    "run component": encoder.similarity(merged_pointer, encoder.encode(run_type)),
    "man2 component": encoder.similarity(merged_pointer, encoder.encode(man_type_2)),
    "man/run":encoder.similarity(encoder.encode(man_rec_type), encoder.encode(run_rec_type)),
    "unrelated {y : Ind}": encoder.similarity(
        merged_pointer, encoder.encode(RecType({"y": Ind}))
    ),
}
scores

{'man component': 0.7398838996887207,
 'run component': 0.741435170173645,
 'man2 component': 0.4947201609611511,
 'man/run': 0.012400000356137753,
 'unrelated {y : Ind}': -0.01742813177406788}

### Comparing VSA models and dimensions

TorchHD calls the hypervector algebra selection `vsa`; `Codebook` forwards this keyword unchanged. The experiment below compares binary spatter codes (`BSC`), multiply-add-permute vectors (`MAP`), and holographic reduced representations (`HRR`) at several dimensionalities.

In [22]:
results = []
for vsa in ("BSC", "MAP", "HRR"):
    for dimensions in (512, 2_048, 10_000):
        trial = Encoder(Codebook(dimensions=dimensions, vsa=vsa, seed=7))
        merged = trial.encode(merged_type)
        man_similarity = trial.similarity(merged, trial.encode(man_type))
        run_similarity = trial.similarity(merged, trial.encode(run_type))
        unrelated_similarity = trial.similarity(
            merged, trial.encode(RecType({"y": Ind}))
        )
        assert man_similarity > unrelated_similarity
        assert run_similarity > unrelated_similarity
        results.append({
            "vsa": vsa,
            "dimensions": dimensions,
            "man component": round(man_similarity, 3),
            "run component": round(run_similarity, 3),
            "unrelated": round(unrelated_similarity, 3),
        })

results

[{'vsa': 'BSC',
  'dimensions': 512,
  'man component': 0.129,
  'run component': 0.234,
  'unrelated': 0.055},
 {'vsa': 'BSC',
  'dimensions': 2048,
  'man component': 0.143,
  'run component': 0.223,
  'unrelated': -0.013},
 {'vsa': 'BSC',
  'dimensions': 10000,
  'man component': 0.165,
  'run component': 0.21,
  'unrelated': -0.017},
 {'vsa': 'MAP',
  'dimensions': 512,
  'man component': 0.714,
  'run component': 0.723,
  'unrelated': -0.076},
 {'vsa': 'MAP',
  'dimensions': 2048,
  'man component': 0.746,
  'run component': 0.744,
  'unrelated': -0.012},
 {'vsa': 'MAP',
  'dimensions': 10000,
  'man component': 0.74,
  'run component': 0.741,
  'unrelated': -0.017},
 {'vsa': 'HRR',
  'dimensions': 512,
  'man component': 0.682,
  'run component': 0.714,
  'unrelated': -0.008},
 {'vsa': 'HRR',
  'dimensions': 2048,
  'man component': 0.729,
  'run component': 0.718,
  'unrelated': -0.019},
 {'vsa': 'HRR',
  'dimensions': 10000,
  'man component': 0.752,
  'run component': 0.732,
 

Each row is a separate seeded codebook, so the comparison isolates the VSA model and dimensionality. The merged type should have a substantially higher similarity to both of its components than to the unrelated type. This is the first approximate SPA invariant; later examples can build on it for cleanup, memory, and question answering.

In [23]:
encoder.codebook._pointers.keys()

dict_keys(['label:x', 'type:Ind', 'label:c_man', 'dependency:x', 'role:dependent-function', 'role:bound-variable', 'variable:v', 'role:domain', 'role:body', 'role:predicate', 'predicate:man', 'role:argument:0', 'role:dependencies', 'label:c_run', 'predicate:run', 'label:y', 'dependency:y'])

In [24]:
chase = Pred("chase", [Ind, Ind])
question = Fun("v", Ind, PType(chase, ["dog52", "v"]))
answer = "cat43"
proposition = question.app(answer)

proposition.show()

'chase(dog52, cat43)'

# Question answering from SPA-TTR memory

This section implements the one-place QA path proposed by Larsson, Ginzburg, Cooper, and Lücking, [*Finding Answers to Questions: Bridging between Type-based and Computational Neuroscience Approaches*](https://aclanthology.org/2025.iwcs-main.11/). The paper connects TTR questions, represented as functions, to SPA operations over structured semantic pointers.

The basic example is:

$$q = \lambda v : Ind.\; chase(dog52, v), \qquad a = cat43, \qquad q(a) = chase(dog52, cat43).$$

The preceding cell constructs this underlying lambda representation using existing categorical TTR. `pyttr` stores the abstracted variable explicitly; the paper’s SPA representation instead marks its position with an identity vector and records the path to that position.

## Why a dedicated `QuestionEncoder`

The current `Encoder.encode(Fun(...))` is a general structural encoding of a TTR function: it binds a variable, its domain, and its body. That is appropriate for representing a TTR function, but it is not the question representation needed by the paper’s answer-retrieval algorithm.

For a one-place wh-question, the paper encodes a question `Q` with three fields:

- `lambdapath` bound to the role holding the answer, here `arg2`;
- `lambdatype` bound to `Ind`; and
- `body` bound to the proposition pattern `pred ⊛ chase + arg1 ⊛ dog52 + arg2 ⊛ I`, where `I` is the HRR identity vector.

The identity vector marks the abstracted position without introducing a normal filler. The explicit path tells the retrieval mechanism where to unbind the answer. These fields are QA-specific, so adding a dedicated `QuestionEncoder` avoids changing the meaning of generic TTR function encoding.

## First implementation checkpoint: application and extraction

With `Q` as above and answer pointer `A = cat43`, the paper’s SPA function application replaces the identity-filled answer position with `A`:

$$Path = Clean_{Paths}(Q \mathbin{\circledast} lambdapath'),$$
$$apply(Q, A) = Q \mathbin{\circledast} body' - Path + Path \mathbin{\circledast} A.$$

`QuestionEncoder.apply(Q, cat43)` compares the result with the independently encoded `chase(dog52, cat43)`. In the reverse direction, given a known matching proposition `P`, `QuestionEncoder.extract(Q, P)` performs answer extraction:

$$extract(Q, P) = P \mathbin{\circledast} (Q \mathbin{\circledast} lambdapath')'.$$

After cleanup against known individual pointers, this should select `cat43`. Both tests use similarity rankings and margins, not vector equality.

## Why a dedicated `AssociativeMemory`

Extraction assumes that the matching proposition `P` is already known. The central extra task in the paper is locating `P` from a question body in long-term memory. A generic vector collection is insufficient because this step needs stable memory indices, cleanup over those indices, and cleanup over the stored proposition pointers.

The paper models memory as a superposition of indexed ptypes:

$$M = 1 \mathbin{\circledast} P_1 + 2 \mathbin{\circledast} P_2 + \ldots + n \mathbin{\circledast} P_n.$$

For a question body cue `B = Q ⊛ body'`, `AssociativeMemory` estimates the matching index, cleans it against registered indices, then recovers and cleans up the proposition:

$$Search_M(B) = M \mathbin{\circledast} (Clean_{Nat}(M \mathbin{\circledast} B'))'.$$

`QuestionEncoder` and `AssociativeMemory` therefore have distinct responsibilities: the former produces a linguistically motivated query and answer path; the latter finds and cleans up a proposition that satisfies the query.

## Target toy model

The paper’s proof-of-concept stores six propositions:

```text
dog chases cat       dog chases cow
dog sees mouse       mouse sees cow
cat likes mouse      dog likes cow
```

The initial target questions and expected answer sets are:

| Question | Matching proposition | Answer |
| --- | --- | --- |
| Who does the cat like? | `cat likes mouse` | `mouse` |
| Who does the dog chase? | `dog chases cat`, `dog chases cow` | `cat` or `cow` |
| Who likes the cow? | `dog likes cow` | `dog` |

The second question is deliberately ambiguous. `AssociativeMemory.ranked_propositions()` returns the competing proposition candidates; the demonstration below retains that ranking rather than treating the first vector match as an unambiguous answer.

## Backend and test policy

The record-type merge experiment can compare several TorchHD VSA models. The QA implementation uses `vsa="HRR"`, because the paper’s application, extraction, and memory equations rely on circular convolution and its approximate inverse. Equivalent operations in other VSA models can be evaluated separately.

The paper reports that its 128-dimensional neural proof-of-concept needs memory cleanup: without it, accumulated unbinding noise can select the wrong answer. This non-neural TorchHD implementation uses 10,000 dimensions for stable regression tests, fixes the codebook seed, and checks rankings rather than vector equality.

Implemented here: (1) `QuestionEncoder` and direct application/extraction; (2) `AssociativeMemory` lookup and proposition cleanup; and (3) the six-proposition toy model with ranked ambiguity. Answer typechecking against the `lambdatype` field remains the next extension.

## The toy model in categorical TTR

First, construct the six ptypes and the three questions with ordinary categorical `pyttr`. Function application is exact: applying a candidate answer produces a ptype. For this small comparison, matching is exact equality of the displayed ptype; this is a transparent symbolic stand-in for locating a proposition in a finite knowledge base, rather than a new categorical-TTR memory mechanism.

In [25]:
see = Pred("see", [Ind, Ind])
like = Pred("like", [Ind, Ind])

toy_propositions = {
    "p1": PType(chase, ["dog", "cat"]),
    "p2": PType(chase, ["dog", "cow"]),
    "p3": PType(see, ["dog", "mouse"]),
    "p4": PType(see, ["mouse", "cow"]),
    "p5": PType(like, ["cat", "mouse"]),
    "p6": PType(like, ["dog", "cow"]),
}
candidates = ("dog", "cat", "cow", "mouse")

categorical_questions = {
    "Who does the cat like?": Fun("v", Ind, PType(like, ["cat", "v"])),
    "Who does the dog chase?": Fun("v", Ind, PType(chase, ["dog", "v"])),
    "Who likes the cow?": Fun("v", Ind, PType(like, ["v", "cow"])),
}

known_ptypes = {ptype.show() for ptype in toy_propositions.values()}
def categorical_answers(question):
    return [
        answer for answer in candidates
        if question.app(answer).show() in known_ptypes
    ]

{text: categorical_answers(question) for text, question in categorical_questions.items()}

{'Who does the cat like?': ['mouse'],
 'Who does the dog chase?': ['cat', 'cow'],
 'Who likes the cow?': ['dog']}

## The same toy model in HRR-SPA

The SPA version stores the same categorical ptypes as indexed HRR bindings. `QuestionEncoder` maps each TTR function to the paper’s identity-and-path representation. `AssociativeMemory` retrieves a proposition using the incomplete question body, cleans that proposition against the stored ptypes, unbinds the answer role, and cleans the result against the four individual candidates.

In [26]:
qa_encoder = QuestionEncoder(Codebook(dimensions=10_000, vsa="HRR", seed=7))
memory = AssociativeMemory(qa_encoder)
for label, ptype in toy_propositions.items():
    memory.add(label, ptype)

spa_results = {}
for text, question in categorical_questions.items():
    encoded_question = qa_encoder.encode(question)
    match, answer, answer_scores = memory.answer(encoded_question, candidates)
    spa_results[text] = {
        "retrieved proposition": match.label,
        "answer": answer,
        "ranked propositions": memory.ranked_propositions(encoded_question)[:2],
        "ranked answers": answer_scores,
    }

spa_results

{'Who does the cat like?': {'retrieved proposition': 'p5',
  'answer': 'mouse',
  'ranked propositions': (('p5', 0.6669754385948181),
   ('p6', 0.34460312128067017)),
  'ranked answers': (('mouse', 0.49466991424560547),
   ('cow', 0.008281310088932514),
   ('dog', -0.007507105357944965),
   ('cat', -0.008528873324394226))},
 'Who does the dog chase?': {'retrieved proposition': 'p2',
  'answer': 'cow',
  'ranked propositions': (('p1', 0.6697357892990112),
   ('p2', 0.6641653180122375)),
  'ranked answers': (('cow', 0.4944166839122772),
   ('dog', 0.011945786885917187),
   ('mouse', 0.003569096326828003),
   ('cat', -0.003980484791100025))},
 'Who likes the cow?': {'retrieved proposition': 'p6',
  'answer': 'dog',
  'ranked propositions': (('p6', 0.6647118926048279),
   ('p5', 0.3353828191757202)),
  'ranked answers': (('dog', 0.49367979168891907),
   ('cat', 0.013433901593089104),
   ('cow', -0.002351282397285104),
   ('mouse', -0.010083546862006187))}}

The categorical and SPA results agree on the two unambiguous questions. For *Who does the dog chase?*, categorical TTR returns both well-formed matches, while the SPA memory exposes the two top proposition candidates and selects its highest-scoring retrieval for the single cleanup path. This is why the ranked propositions remain part of the result.